# DecisionRAG: public-data loader and tiny navigation demo

This notebook accompanies [Wan et al., arXiv:2610.35868](https://arxiv.org/abs/2610.35868). It loads a public benchmark split used by the paper and implements the hierarchy-navigation skeleton without requiring the proprietary Jev decision-model API.

In [ ]:
!pip -q install datasets

import re
from pprint import pprint

from datasets import load_dataset

## 1. Load a public benchmark slice

The paper evaluates the DAPR versions of ConditionalQA and Natural Questions. Loading only the ConditionalQA query split keeps this notebook small; the full corpora contain tens of thousands to millions of passages.

In [ ]:
queries = load_dataset(
    'UKPLab/dapr',
    'ConditionalQA-queries',
    split='test',
)

print(f'Loaded {len(queries):,} DAPR ConditionalQA test queries.')
for row in queries.select(range(min(2, len(queries)))):
    pprint(row)
    print()

## 2. Build a tiny corpus hierarchy

Every node has a short description and optional children. Leaves are passages. This mirrors the paper's cluster → page → paragraph or help-center → page → passage structures.

In [ ]:
corpus = {
    'Films': {
        'description': 'Film pages about directors, main characters, release years, and story details.',
        'children': {
            'Memento (film)': {
                'description': 'Christopher Nolan film released in 2000.',
                'children': {
                    'memento-plot': "Christopher Nolan's 2000 film Memento has Leonard Shelby as its main character.",
                    'memento-production': 'Christopher Nolan wrote and directed Memento, released in 2000.',
                },
            },
            'Gladiator (2000 film)': {
                'description': 'A 2000 historical film directed by Ridley Scott.',
                'children': {
                    'gladiator-plot': 'Gladiator follows the Roman general Maximus Decimus Meridius.',
                },
            },
            'Christopher Nolan': {
                'description': 'Film director known for Memento and later blockbusters.',
                'children': {
                    'nolan-career': 'Nolan gained recognition with Memento, whose protagonist is Leonard Shelby.',
                },
            },
        },
    },
    'Public services': {
        'description': 'Government benefits, licenses, taxes, and public services.',
        'children': {
            'Licensing': {
                'description': 'Rules for driving licenses and vehicle registration.',
                'children': {
                    'license-address': 'Drivers must report an address change within ten days.',
                },
            },
        },
    },
}

query = "Who is the main character in Christopher Nolan's film from the year 2000?"

## 3. Navigate with irreversible top-k pruning

The paper supplies all candidates at a level to a decision model and keeps only the best branches. Here, token overlap is a simple stand-in scorer. Once a branch is removed, its descendants are never evaluated.

In [ ]:
def token_set(text):
    return set(re.findall(r'[a-z0-9]+', text.lower()))

def score_candidates(query, descriptions):
    # Transparent Jaccard overlap used only as a stand-in relevance score.
    query_tokens = token_set(query)
    return [
        len(query_tokens & token_set(text)) / max(1, len(query_tokens | token_set(text)))
        for text in descriptions
    ]

def rank_nodes(query, nodes):
    names = list(nodes)
    descriptions = [
        nodes[name]['description'] if isinstance(nodes[name], dict) else nodes[name]
        for name in names
    ]
    scores = score_candidates(query, descriptions)
    return sorted(zip(names, scores), key=lambda item: item[1], reverse=True)

# Stage 1: retain one top-level branch.
topic_ranking = rank_nodes(query, corpus)
kept_topic = topic_ranking[0][0]

# Stage 2: retain the two most promising pages below that branch.
pages = corpus[kept_topic]['children']
page_ranking = rank_nodes(query, pages)
kept_pages = [name for name, _ in page_ranking[:2]]

# Stage 3: pool and rank only passages under retained pages.
passages = {
    passage_id: text
    for page_name in kept_pages
    for passage_id, text in pages[page_name]['children'].items()
}
passage_ranking = rank_nodes(query, passages)

print('Query:', query)
print('Topic ranking:', topic_ranking)
print('Kept topic:', kept_topic)
print('Page ranking:', page_ranking)
print('Kept pages:', kept_pages)
print('Final passage ranking:')
for passage_id, score in passage_ranking:
    print(f'  {score:.3f}  {passage_id}: {passages[passage_id]}')

### Scope

This demonstrates DecisionRAG's control flow, not its learned relevance judgments or reported accuracy. A faithful reproduction needs the authors' unshared hierarchy artifacts and routing code, plus hosted Jev access or trained Qwen3 decision-model checkpoints.